In [1]:
%pip install torch datasets tokenizers sacremoses

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import autocast, GradScaler
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import math
import os
import random
import numpy as np
import time

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

SRC_LANG = 'de'
TGT_LANG = 'en'
MAX_LEN = 128
VOCAB_SIZE = 37000
BATCH_SIZE = 256

Using device: cuda
GPU Name: NVIDIA A100-SXM4-80GB


In [3]:
BATCH_SIZE = 256

In [5]:
NUM_SAMPLES = 2000000

print("Loading WMT16 dataset...")
raw_dataset = load_dataset("wmt16", "de-en")

print(f"Original training size: {len(raw_dataset['train'])}")

train_subset = raw_dataset['train'].shuffle(seed=SEED).select(range(NUM_SAMPLES))
valid_subset = raw_dataset['validation'] 
test_subset = raw_dataset['test']

print(f"Subset training size: {len(train_subset)}")

def batch_iterator(batch_size=10000):
    for i in range(0, len(train_subset), batch_size):
        batch = train_subset[i : i + batch_size]
        for example in batch['translation']:
            yield example[SRC_LANG]
            yield example[TGT_LANG]

print("Training BPE Tokenizer on subset...")
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE, 
    special_tokens=["<unk>", "<pad>", "<sos>", "<eos>"],
    min_frequency=2,
    show_progress=True
)

tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

UNK_IDX = tokenizer.token_to_id("<unk>")
PAD_IDX = tokenizer.token_to_id("<pad>")
SOS_IDX = tokenizer.token_to_id("<sos>")
EOS_IDX = tokenizer.token_to_id("<eos>")

print(f"Tokenizer trained. Vocab size: {tokenizer.get_vocab_size()}")

Loading WMT16 dataset...
Original training size: 4548885
Subset training size: 2000000
Training BPE Tokenizer on subset...



Tokenizer trained. Vocab size: 37000


In [6]:
def collate_batch(batch):
    src_batch, tgt_batch = [], []
    
    for item in batch:
        pair = item['translation']
        src_enc = tokenizer.encode(pair[SRC_LANG]).ids
        tgt_enc = tokenizer.encode(pair[TGT_LANG]).ids
        
        src_enc = src_enc[:MAX_LEN-2]
        tgt_enc = tgt_enc[:MAX_LEN-2]
        
        src_tensor = torch.tensor([SOS_IDX] + src_enc + [EOS_IDX], dtype=torch.long)
        tgt_tensor = torch.tensor([SOS_IDX] + tgt_enc + [EOS_IDX], dtype=torch.long)
        
        src_batch.append(src_tensor)
        tgt_batch.append(tgt_tensor)
        
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch

train_loader = data.DataLoader(train_subset, batch_size=BATCH_SIZE, 
                               shuffle=True, collate_fn=collate_batch, 
                               num_workers=4, pin_memory=True)

valid_loader = data.DataLoader(valid_subset, batch_size=BATCH_SIZE, 
                               shuffle=False, collate_fn=collate_batch, 
                               num_workers=4, pin_memory=True)

test_loader = data.DataLoader(test_subset, batch_size=BATCH_SIZE, 
                              shuffle=False, collate_fn=collate_batch, 
                              num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")

Train batches: 7813


In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_model // n_head
        self.n_head = n_head
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([self.d_k])).to(device)

    def forward(self, q, k, v, mask=None):
        batch_size = q.shape[0]
        Q = self.w_q(q).view(batch_size, -1, self.n_head, self.d_k).permute(0, 2, 1, 3)
        K = self.w_k(k).view(batch_size, -1, self.n_head, self.d_k).permute(0, 2, 1, 3)
        V = self.w_v(v).view(batch_size, -1, self.n_head, self.d_k).permute(0, 2, 1, 3)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        attention = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attention), V)
        x = x.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)
        return self.fc(x)

class PositionwiseFeedforward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.pff = PositionwiseFeedforward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.pff(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.enc_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.pff = PositionwiseFeedforward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, trg_mask)))
        x = self.norm2(x + self.dropout(self.enc_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.pff(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_layers=6, n_head=8, d_ff=2048, max_len=200, dropout=0.1):
        super().__init__()
        self.encoder_emb = nn.Embedding(vocab_size, d_model)
        self.decoder_emb = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = PositionalEncoding(d_model, max_len, dropout)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, n_head, d_ff, dropout) for _ in range(n_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, n_head, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.scale = torch.sqrt(torch.FloatTensor([d_model])).to(device)
        self.dropout = nn.Dropout(dropout)

    def make_src_mask(self, src):
        return (src != PAD_IDX).unsqueeze(1).unsqueeze(2)

    def make_trg_mask(self, trg):
        trg_pad_mask = (trg != PAD_IDX).unsqueeze(1).unsqueeze(2)
        trg_len = trg.shape[1]
        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=device)).bool()
        return trg_pad_mask & trg_sub_mask

    def encode(self, src, src_mask):
        x = self.encoder_emb(src) * self.scale
        x = self.pos_embedding(x)
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, trg, enc_out, src_mask, trg_mask):
        x = self.decoder_emb(trg) * self.scale
        x = self.pos_embedding(x)
        for layer in self.decoder_layers:
            x = layer(x, enc_out, src_mask, trg_mask)
        return x

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_src = self.encode(src, src_mask)
        output = self.decode(trg, enc_src, src_mask, trg_mask)
        return self.fc_out(output)

In [8]:
# Hyperparameters for Transformer BASE
D_MODEL = 512
N_LAYERS = 6
N_HEADS = 8
D_FF = 2048
DROPOUT = 0.1
EPOCHS = 15
LEARNING_RATE = 0.0001

vocab_size = tokenizer.get_vocab_size()
model = Transformer(vocab_size, D_MODEL, N_LAYERS, N_HEADS, D_FF, MAX_LEN, DROPOUT).to(device)

def initialize_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)

model.apply(initialize_weights)
print(f'The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

The model has 101,007,496 trainable parameters


In [9]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 2))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
            true_dist[:, PAD_IDX] = 0
            mask = torch.nonzero(target.data == PAD_IDX)
            if mask.dim() > 0:
                true_dist.index_fill_(0, mask.squeeze(), 0.0)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))

criterion = LabelSmoothingLoss(classes=vocab_size, smoothing=0.1).to(device)
scaler = GradScaler()

In [10]:
def train_epoch(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            output = model(src, trg[:, :-1])
            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)
            loss = criterion(output, trg)
        
        scaler.scale(loss).backward()
        
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
        if i % 100 == 0:
            print(f"Step {i}/{len(iterator)} | Loss: {loss.item():.4f}")
            
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)
            with autocast():
                output = model(src, trg[:, :-1])
                output_dim = output.shape[-1]
                output = output.contiguous().view(-1, output_dim)
                trg = trg[:, 1:].contiguous().view(-1)
                loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

In [ ]:
CLIP = 1
best_valid_loss = float('inf')

for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_loader, criterion)
    
    end_time = time.time()
    mins = int((end_time - start_time) / 60)
    secs = int((end_time - start_time) % 60)
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'wmt16_model_final.pt')
    
    print(f'Epoch: {epoch+1:02} | Time: {mins}m {secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} | PPL: {math.exp(valid_loss):7.3f}')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step 0/7813 | Loss: 2.5009
Step 100/7813 | Loss: 2.3520
Step 200/7813 | Loss: 2.0775
Step 300/7813 | Loss: 1.9759
Step 400/7813 | Loss: 2.2964
Step 600/7813 | Loss: 1.6025
Step 7800/7813 | Loss: 1.2454


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch: 01 | Time: 29m 44s
	Train Loss: 1.543 | PPL:   4.678
	 Val. Loss: 1.900 | PPL:   6.684


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step 0/7813 | Loss: 1.3181
Step 100/7813 | Loss: 1.1248
Step 200/7813 | Loss: 1.2214
Step 300/7813 | Loss: 1.4869
Step 400/7813 | Loss: 1.6804
Step 4200/7813 | Loss: 1.2244


In [21]:
def translate_sentence(sentence, model, max_len=50):
    model.eval()
    # Tokenize
    tokens = tokenizer.encode(sentence).ids
    src_tensor = torch.LongTensor([SOS_IDX] + tokens + [EOS_IDX]).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src_tensor)
    
    with torch.no_grad():
        enc_src = model.encode(src_tensor, src_mask)

    trg_indexes = [SOS_IDX]

    for i in range(max_len):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)
        trg_mask = model.make_trg_mask(trg_tensor)
        
        with torch.no_grad():
            output = model.decode(trg_tensor, enc_src, src_mask, trg_mask)
            output = model.fc_out(output)
        
        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == EOS_IDX:
            break
    
    decoded = tokenizer.decode(trg_indexes, skip_special_tokens=True)
    return decoded

model.load_state_dict(torch.load('wmt16_model_final.pt'))

<All keys matched successfully>

In [22]:
german_sentences = [
    "Ich bin Student.",
    "Das Haus ist groß.",
    "Er trinkt Wasser.",
    "Wir gehen heute einkaufen.",
    "Sie liest ein Buch.",
    "Der Himmel ist blau.",
    "Ich habe Hunger.",
    "Das Auto fährt schnell.",
    "Sie spielt Klavier.",
    "Es regnet heute.",
    "Ich komme aus Deutschland.",
    "Der Kaffee ist heiß.",
    "Wir lernen Deutsch.",
    "Die Katze schläft.",
    "Er arbeitet im Büro.",
    "Das Essen schmeckt gut.",
    "Ich sehe einen Film.",
    "Sie wohnt in Berlin.",
    "Der Hund bellt.",
    "Ich verstehe das nicht.",
    "Die Blume ist schön.",
    "Er fährt mit dem Fahrrad.",
    "Wir machen Urlaub.",
    "Das Kind lacht.",
    "Ich brauche Hilfe."
]
for text in german_sentences:
    translation = translate_sentence(text, model)
    print(translation)

I am a student .
The House is large .
He at water .
We are going to buy today .
She reads a book .
The sky is blue .
I have hunger .
The car is fast .
She plays piano .
It is currently ra ining .
I come from Germany .
The coffee is hot .
We learn German .
The cat sle e ps .
He works in the office .
The food tastes good .
I see a movie .
She lives in Berlin .
The dog is sh ining .
I do not understand that .
The flower is beautiful .
He drives by bicycle .
We are a holiday destination .
The baby laugh s .
I need help .


In [17]:
%pip install sacrebleu evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [evaluate]1/2 [evaluate]

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [23]:
import evaluate
from tqdm import tqdm
from datasets import load_dataset

N_EVAL = 500

if 'test_subset' in globals():
    source_data = test_subset
elif 'dataset' in globals():
    source_data = dataset['test']
else:
    source_data = load_dataset("wmt16", "de-en", split='test')

test_data_small = source_data.select(range(N_EVAL))

print(f"Starting evaluation on first {len(test_data_small)} samples...")

bleu = evaluate.load("sacrebleu")
model.eval()

predictions = []
references = []

def translate_single(text):
    ids = tokenizer.encode(text).ids
    src_tensor = torch.LongTensor([SOS_IDX] + ids + [EOS_IDX]).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src_tensor)
    
    with torch.no_grad():
        enc_src = model.encode(src_tensor, src_mask)
    
    trg_indexes = [SOS_IDX]
    for i in range(100):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)
        trg_mask = model.make_trg_mask(trg_tensor)
        
        with torch.no_grad():
            output = model.decode(trg_tensor, enc_src, src_mask, trg_mask)
            output = model.fc_out(output)
        
        pred_token = output.argmax(2)[:, -1].item()
        
        if pred_token == EOS_IDX:
            break
        trg_indexes.append(pred_token)
    decoded = tokenizer.decode(trg_indexes, skip_special_tokens=True)
    
    decoded = decoded.replace(" .", ".").replace(" ,", ",").replace(" ?", "?").replace(" !", "!")
    return decoded

for item in tqdm(test_data_small):
    pair = item['translation'] if 'translation' in item else item
    
    src_text = pair[SRC_LANG]
    tgt_text = pair[TGT_LANG]
    
    pred_text = translate_single(src_text)
    
    predictions.append(pred_text)
    references.append([tgt_text])

results = bleu.compute(predictions=predictions, references=references, force=True)

print("\n" + "="*30)
print(f"BLEU Score (on {N_EVAL} samples): {results['score']:.2f}")
print("="*30)

Starting evaluation on first 500 samples...


100%|██████████| 500/500 [01:18<00:00,  6.34it/s]



BLEU Score (on 500 samples): 24.14
